In [2]:
import pandas as pd
from pathlib import Path
import sqlite3

In [3]:
# making a connection to the database
cwd = Path.cwd()
project_root = cwd.parent if cwd.name.lower() == "notebooks" else cwd
db_dir = project_root / "data" / "raw"
db_dir.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(db_dir / "stroke.db")

In [4]:
raw = pd.read_sql("SELECT * FROM stroke", con)
print(f"LOADED: {raw.shape[0]} rows, {raw.shape[1]} columns.")

LOADED: 6053 rows, 27 columns.


general info:

In [5]:
raw.shape

(6053, 27)

In [6]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 6053 entries, 0 to 6052
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   SEQN                      6053 non-null   int64  
 1   gender                    6053 non-null   str    
 2   age                       6053 non-null   int64  
 3   race                      6053 non-null   str    
 4   weight_kg                 5972 non-null   float64
 5   height_cm                 5986 non-null   float64
 6   bmi                       5959 non-null   float64
 7   bmi_category              5959 non-null   str    
 8   education                 6049 non-null   str    
 9   marital_status            6049 non-null   str    
 10  employment_status         6052 non-null   str    
 11  diabetes                  6053 non-null   float64
 12  hypertension              6049 non-null   float64
 13  high_cholesterol          6018 non-null   float64
 14  congestive_heart_fa

missing value investigation

In [7]:
raw.shape[0]

6053

In [8]:
round(raw.isnull().sum() / raw.shape[0] * 100, 2)

SEQN                         0.00
gender                       0.00
age                          0.00
race                         0.00
weight_kg                    1.34
height_cm                    1.11
bmi                          1.55
bmi_category                 1.55
education                    0.07
marital_status               0.07
employment_status            0.02
diabetes                     0.00
hypertension                 0.07
high_cholesterol             0.58
congestive_heart_failure     0.13
coronary_heart_disease       0.36
angina                       0.25
heart_attack                 0.13
cancer                       0.10
thyroid_problem              0.18
liver_condition              0.17
kidney_disease               0.18
sleep_duration_hours         1.04
sleep_bedtime                1.60
smoking_status               0.07
alcohol_frequency           27.24
stroke                       0.00
dtype: float64

In [9]:
missing_summary = (
    pd.DataFrame({
        "n_missing": raw.isna().sum(),
        "pct_missing": raw.isna().mean() * 100,
    })
    .sort_values("pct_missing", ascending=False)
)
missing_summary

,n_missing,pct_missing
alcohol_frequency,1649,27.242690
sleep_bedtime,97,1.602511
bmi,94,1.552949
bmi_category,94,1.552949
weight_kg,81,1.338179
height_cm,67,1.106889
sleep_duration_hours,63,1.040806
high_cholesterol,35,0.578226
coronary_heart_disease,22,0.363456
angina,15,0.247811


In [10]:
print(f"Rows remaining if you all rows with missing values: "
    f"{raw.dropna().shape[0]} of {len(raw)}")

Rows remaining if you all rows with missing values: 4227 of 6053


In [11]:
print("Most common missingness patterns:")
print(raw.isna().value_counts().head(10))

Most common missingness patterns:
SEQN   gender  age    race   weight_kg  height_cm  bmi    bmi_category  education  marital_status  employment_status  diabetes  hypertension  high_cholesterol  congestive_heart_failure  coronary_heart_disease  angina  heart_attack  cancer  thyroid_problem  liver_condition  kidney_disease  sleep_duration_hours  sleep_bedtime  smoking_status  alcohol_frequency  stroke
False  False   False  False  False      False      False  False         False      False           False              False     False         False             False                     False                   False   False         False   False            False            False           False                 False          False           False              False     4227
                                                                                                                                                                                                                            

In [12]:
print(raw.drop(columns=['alcohol_frequency']).isna().value_counts())

SEQN   gender  age    race   weight_kg  height_cm  bmi    bmi_category  education  marital_status  employment_status  diabetes  hypertension  high_cholesterol  congestive_heart_failure  coronary_heart_disease  angina  heart_attack  cancer  thyroid_problem  liver_condition  kidney_disease  sleep_duration_hours  sleep_bedtime  smoking_status  stroke
False  False   False  False  False      False      False  False         False      False           False              False     False         False             False                     False                   False   False         False   False            False            False           False                 False          False           False     5744
                                                                                                                                                                                                                                                                                                  Tr

If you keep alcohol frequency out of it, then the combination of sleep duration and sleep bedtime are an interesting feature combination that is linked to missingness. I could see someone not knowing how long they sleep and when because it is so irregular. an insomniac or something. There are also 47 rows with missing weight and height and bmi. makes sense, especially the BMI one. then it's a history of high cholesterol but idk maybe a lot of people just didn't know. then weight not being there and bmi neither. again makes sense bc bmi is a calculation of. 

In [13]:
print(raw.isna().sum(axis=1).value_counts().sort_index())

0    4227
1    1610
2      98
3      42
4      32
5      39
6       3
7       1
8       1
Name: count, dtype: int64


In [14]:
measurements = ["weight_kg", "height_cm", "bmi"]
print(raw[measurements].isna().value_counts().rename("rows"))

weight_kg  height_cm  bmi  
False      False      False    5959
True       True       True       54
           False      True       27
False      True       True       13
Name: rows, dtype: int64


In [15]:
measurements = ["weight_kg", "height_cm", "bmi"]
print(raw[measurements].isna().value_counts().rename("rows"))

both_measured = raw["weight_kg"].notna() & raw["height_cm"].notna()
print("BMI missing despite both measurements:",
      (raw["bmi"].isna() & both_measured).sum())

weight_kg  height_cm  bmi  
False      False      False    5959
True       True       True       54
           False      True       27
False      True       True       13
Name: rows, dtype: int64
BMI missing despite both measurements: 0


In [16]:
both_measured = raw["weight_kg"].notna() & raw["height_cm"].notna()
print("BMI missing despite weight and height being there:",
    (raw["bmi"].isna() & both_measured).sum())

BMI missing despite weight and height being there: 0


Checking out if missingness differ per target variable

In [19]:
features = [
    "weight_kg", "height_cm", "bmi",
    "sleep_duration_hours", "sleep_bedtime", "high_cholesterol"
]

missing_by_stroke = raw.groupby("stroke")[features].agg(
    lambda x: x.isna().mean() * 100
)

missing_by_stroke.round(1)

,weight_kg,height_cm,bmi,sleep_duration_hours,sleep_bedtime,high_cholesterol
stroke,,,,,,
0,1.2,1.0,1.4,1.0,1.5,0.6
1,3.9,2.5,3.9,1.4,3.2,0.4


there is a small difference in missingness between the non-strokies and strokies.

In [22]:
print("Rows per stroke group:")
print(raw.groupby("stroke").size())

print("\nMissing values per feature and stroke group:")
print(raw.groupby("stroke")[features].agg(lambda x: x.isna().sum()))

Rows per stroke group:
stroke
0    5773
1     280
dtype: int64

Missing values per feature and stroke group:
        weight_kg  height_cm  bmi  sleep_duration_hours  sleep_bedtime  \
stroke                                                                   
0              70         60   83                    59             88   
1              11          7   11                     4              9   

        high_cholesterol  
stroke                    
0                     34  
1                      1  


In [23]:
pd.crosstab(raw["stroke"], raw["bmi"].isna())

bmi,False,True
stroke,,
0,5690,83
1,269,11


BMI is missing in 1.4% of non-stroke records and 3.9% of stroke records. Missing BMI occurs when height or weight is missing, so investigate those source measurements; don’t remove rows based on this table alone

In [29]:
age_group = pd.cut(
    raw["age"],
    bins = [19, 39, 59, 80],
    labels = ["20-39", "40-59", "60-80"]
)

by_age = raw.assign(age_group = age_group).groupby("age_group").agg(
    rows=("age", "size"),
    weight_missing = ("weight_kg", lambda x: x.isna().sum()),
    height_missing = ("height_cm", lambda x: x.isna().sum())
)

by_age["weight_missing_pct"] = 100.0 * by_age["weight_missing"] / by_age["rows"]
by_age["height_missing_pct"] = 100.0* by_age["height_missing"] / by_age["rows"]

by_age.round(2)

,rows,weight_missing,height_missing,weight_missing_pct,height_missing_pct
age_group,,,,,
20-39,1559,19,19,1.22,1.22
40-59,1728,22,17,1.27,0.98
60-80,2766,40,31,1.45,1.12


weight is less often recorded in the oldest age group and the youngest age group has the lowest amount of missingness. for height that's the opposite the youngest group has the highest missingness. the middle aged group has the lowest amount of missingness in height.

In [30]:
by_gender = raw.groupby("gender").agg(
    rows = ("gender", "size"),
    weight_missing = ("weight_kg", lambda x: x.isna().sum()),
    height_missing = ("height_cm", lambda x: x.isna().sum())
)

by_gender["weight_missing_pct"] = 100.0 * by_gender["weight_missing"] / by_gender["rows"]
by_gender["height_missing_pct"] = 100.0* by_gender["height_missing"] / by_gender["rows"]

by_gender.round(2)

,rows,weight_missing,height_missing,weight_missing_pct,height_missing_pct
gender,,,,,
Female,3340,49,37,1.47,1.11
Male,2713,32,30,1.18,1.11


Weight was missing slightly more often among women; height missingness was equal by gender. The reason is unknown

In [43]:
df = raw.copy()
df["alcohol_frequency"] = raw["alcohol_frequency"].fillna("Unclear")

missingness_summary = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "pct_missing": df.isna().mean() * 100.0
}).sort_values("pct_missing", ascending = True)

missingness_summary.round(2)

,n_missing,pct_missing
SEQN,0,0.00
gender,0,0.00
age,0,0.00
race,0,0.00
diabetes,0,0.00
stroke,0,0.00
alcohol_frequency,0,0.00
employment_status,1,0.02
hypertension,4,0.07
smoking_status,4,0.07
